# 🌲 Notebook 3 — The Full Picture: Read-Repair + Anti-Entropy + Hinted Handoff

Read-repair is powerful but has one blind spot: **keys nobody reads never get repaired**. In a large cluster, most keys are cold. To keep *everything* eventually consistent, Dynamo-style databases combine three mechanisms:

| Mechanism | Triggered by | Fixes | Cost |
|---|---|---|---|
| **Hinted handoff** | A write to a dead replica | The recent writes that a replica missed while down | Memory on a peer node |
| **Read-repair** | A client read | Stale values on replicas that *were* queried | A few extra RPCs per read |
| **Anti-entropy** | A scheduled / on-demand sweep | Everything, including cold keys | Background CPU + network |

This notebook simulates all three side by side so you can see how they complement each other.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/read-repair
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
from dataclasses import dataclass, field
from typing import Dict, Tuple, List, Optional
import hashlib, random

Entry = Tuple[str, int]

@dataclass
class Replica:
    name: str
    alive: bool = True
    data: Dict[str, Entry] = field(default_factory=dict)

    def write(self, k, v, ts):
        cur = self.data.get(k)
        if cur is None or ts > cur[1]:
            self.data[k] = (v, ts)

    def read(self, k): return self.data.get(k)


## 1️⃣ Hinted handoff — "I'll remember this write for you"

When a coordinator tries to replicate a write to a replica that's **down**, it stores a *hint* locally: "next time `r3` is alive, deliver this write." When `r3` comes back, the peer flushes the hints. This keeps writes moving during short outages and prevents `r3` from starting out massively behind when it recovers.


In [ ]:
class CoordinatorWithHints:
    def __init__(self, replicas: List[Replica]):
        self.replicas = replicas
        # hints[target_name] = list of pending (k, v, ts)
        self.hints: Dict[str, List[Tuple[str, str, int]]] = {r.name: [] for r in replicas}

    def write(self, k, v, ts):
        for r in self.replicas:
            if r.alive:
                r.write(k, v, ts)
            else:
                # Store a hint for the dead replica.
                self.hints[r.name].append((k, v, ts))

    def deliver_hints(self):
        for r in self.replicas:
            if r.alive and self.hints[r.name]:
                print(f"  📬 delivering {len(self.hints[r.name])} hints to {r.name}")
                for k, v, ts in self.hints[r.name]:
                    r.write(k, v, ts)
                self.hints[r.name].clear()

r1, r2, r3 = Replica("r1"), Replica("r2"), Replica("r3")
co = CoordinatorWithHints([r1, r2, r3])

# r3 goes down for a moment
r3.alive = False
co.write("user:42", "Alice v2", 200)
co.write("user:43", "Bob v1",   201)
print("while r3 down, r3.data =", r3.data, "  (empty)")

# r3 comes back
r3.alive = True
co.deliver_hints()
print("r3.data after hint delivery =", r3.data)


Hints have limits: they expire, they consume memory on the coordinator, and if the node stays down for too long they are dropped. For durable recovery we still need **anti-entropy**.


## 2️⃣ Anti-entropy with a (tiny) Merkle tree

A **Merkle tree** is a tree of hashes that lets two replicas spot-check whether they agree on a range of keys without sending the whole dataset.

- Leaf `i` = hash of `(key, value, ts)` for every key in bucket `i`.
- Internal node = hash of its children.
- If two replicas have the same root hash, their data is identical. If not, they recurse into the subtree that differs.

A real Merkle tree has many levels; here we use a one-level "tree" (just the leaf buckets) to show the *idea*. The interesting part is: **we never send values that already agree.**


In [ ]:
NUM_BUCKETS = 8

def bucket_of(k: str) -> int:
    return int(hashlib.md5(k.encode()).hexdigest(), 16) % NUM_BUCKETS

def bucket_hashes(r: Replica):
    buckets: Dict[int, List[str]] = {i: [] for i in range(NUM_BUCKETS)}
    for k, (v, ts) in sorted(r.data.items()):
        buckets[bucket_of(k)].append(f"{k}={v}@{ts}")
    return {i: hashlib.md5("|".join(items).encode()).hexdigest() for i, items in buckets.items()}

def anti_entropy(a: Replica, b: Replica):
    ha, hb = bucket_hashes(a), bucket_hashes(b)
    diffs = [i for i in range(NUM_BUCKETS) if ha[i] != hb[i]]
    print(f"  buckets that disagree between {a.name} and {b.name}: {diffs}")
    # For each differing bucket, exchange the actual keys and reconcile via LWW.
    for i in diffs:
        keys = {k for r in (a, b) for k in r.data if bucket_of(k) == i}
        for k in keys:
            va, vb = a.read(k), b.read(k)
            if va and (not vb or va[1] > vb[1]): b.write(k, *va)
            elif vb and (not va or vb[1] > va[1]): a.write(k, *vb)
    return len(diffs)

# Seed 50 keys with some drift between r1 and r2.
r1 = Replica("r1"); r2 = Replica("r2")
for i in range(50):
    r1.write(f"k{i}", f"v{i}", 100)
    r2.write(f"k{i}", f"v{i}", 100)
# Simulate r2 missing 3 updates to random keys.
random.seed(7)
for k in random.sample(list(r1.data.keys()), 3):
    r1.write(k, "v-new", 200)       # r1 has the newer value; r2 doesn't.

print("before sweep:", anti_entropy(r1, r2), "buckets disagreed")
print("after sweep :", anti_entropy(r1, r2), "buckets disagreed")


The second sweep reports 0 disagreeing buckets — the replicas have converged. In a real system the sweep runs on a timer (Cassandra: `nodetool repair`) or continuously in the background (Riak: active anti-entropy). The Merkle structure keeps it cheap: you only transfer data for buckets whose hashes differ.


## 3️⃣ Putting it together

A Dynamo-style cluster typically uses **all three** mechanisms:

1. **Hinted handoff** covers the **short downtime** case — a replica blinks offline for 30 seconds; hints catch it up within seconds of its return.
2. **Read-repair** covers the **hot keys** — if a key gets read, it gets fixed.
3. **Anti-entropy** covers the **long tail of cold keys** and the case where hints expired before the replica returned.

None alone is sufficient. Read-repair on hot data keeps latency-sensitive reads fresh. Anti-entropy guarantees *eventual* convergence on everything. Hinted handoff bridges short outages without waiting for the next anti-entropy sweep.


## 📚 Where to look in the real world

- **Cassandra**: `read_repair_chance` (removed in newer versions in favor of blocking/async repair), `nodetool repair` for anti-entropy.
- **DynamoDB / Dynamo paper**: read repair + anti-entropy via Merkle trees + sloppy quorum with hinted handoff.
- **Riak**: active anti-entropy, siblings from vector clocks.
- **ScyllaDB**: similar to Cassandra, with per-table repair scheduling.
- **CockroachDB / Spanner**: avoid this class of problem by using **consensus (Raft/Paxos)** for every write — see the `consensus/` labs.

Next up in this track: the `leader-election` and `consensus` labs, which take the opposite philosophy — prevent replicas from ever disagreeing in the first place.
